In [1]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()

# convert pixels from range 0-16 to approximately 0-1.
X = (digits.data / 16.0).tolist()
y = digits.target.tolist()


X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(len(X_train))
print(len(X_train[0]))
print(y_train[0])

1437
64
5


In [2]:
# 64 pixels → 32 hidden neurons → 10 output neurons
from micrograd.nn import MLP
model = MLP(64, [32, 10])

In [3]:
# check if model setup is complete or not
logits = model(x=X_train[0])
predicted_digit = max(range(10), key=lambda i: logits[i].data)
print("logits: ", logits)
print("predicted_digit: ", predicted_digit)
print("y_train[0]: ", y_train[0])

logits:  [ScalarValue(data=-10.65165590486064, grad=0), ScalarValue(data=15.700851875073583, grad=0), ScalarValue(data=7.089197652923293, grad=0), ScalarValue(data=0.432802666217472, grad=0), ScalarValue(data=3.407233284448372, grad=0), ScalarValue(data=-2.18505955899174, grad=0), ScalarValue(data=2.5917313805292226, grad=0), ScalarValue(data=2.3487833289891893, grad=0), ScalarValue(data=-3.503531036121201, grad=0), ScalarValue(data=-16.462522389063114, grad=0)]
predicted_digit:  1
y_train[0]:  5


In [4]:
def predict(model, x: list[float]) -> int:
    logits = model(x)

    return max(
        range(len(logits)),
        key=lambda index: logits[index].data,
    )

In [ ]:
import random
from micrograd.model_io import save_model
from datetime import datetime

def cross_entropy(logits, target: int):
    # Subtracting a plain numeric maximum improves stability.
    max_logit = max(logit.data for logit in logits)
    shifted = [logit - max_logit for logit in logits]

    exponentials = [logit.exp() for logit in shifted]
    denominator = sum(exponentials)

    probability_of_target = exponentials[target] / denominator

    return -probability_of_target.log()


def batch_loss(
    model,
    batch
):
    losses = []

    for x, target in batch:
        logits = model(x)
        losses.append(cross_entropy(logits, target))

    return sum(losses) / len(losses)


learning_rate = 0.03
batch_size = 32
epochs = 30

training_data = list(zip(X_train, y_train))

for epoch in range(epochs):
    random.shuffle(training_data)

    epoch_loss = 0.0
    batch_count = 0

    for start in range(0, len(training_data), batch_size):
        batch = training_data[start : start + batch_size]

        # Forward pass and loss through Micrograd.
        loss = batch_loss(model, batch)

        # Clear accumulated gradients.
        model.zero_grad()

        # Micrograd backpropagation.
        loss.backward()

        # Manual SGD parameter update.
        for parameter in model.parameters():
            parameter.data -= learning_rate * parameter.grad

        epoch_loss += loss.data
        batch_count += 1

    accuracy = sum(
        predict(model, x) == target
        for x, target in zip(X_test, y_test)
    ) / len(X_test)

    print(
        f"epoch={epoch + 1:02d} "
        f"loss={epoch_loss / batch_count:.4f} "
        f"test_accuracy={accuracy:.2%}"
    )

save_model(
    model = model,
   filepath = f"models/model_{datetime.now().isoformat()}_digit_classifier.json",
    metadata = {
        "classes": list(range(10)),
        "image_width": 8,
        "image_height": 8,
        "pixel_divisor": 16.0,
    },
)

KeyboardInterrupt: 